In [ ]:
# CELL 7
from huggingface_hub import hf_hub_download
from pathlib import Path

ZIP_DIR = Path("/content/models/hynt_zipformer_30m")
ZIP_DIR.mkdir(parents=True, exist_ok=True)

repo_id = "hynt/Zipformer-30M-RNNT-6000h"
files = [
    "tokens.txt",
    "bpe.model",
    "config.json",
    "encoder-epoch-20-avg-10.onnx",
    "decoder-epoch-20-avg-10.onnx",
    "joiner-epoch-20-avg-10.onnx",
]

for f in files:
    try:
        hf_hub_download(repo_id=repo_id, filename=f, local_dir=str(ZIP_DIR))
        print("downloaded:", f)
    except Exception as e:
        print("skip/fail:", f, e)

print("ZIP_DIR =", ZIP_DIR)


In [ ]:
%%bash
# CELL 8
cd /content/

# Gipformer repo
if [ ! -d /content/gipformer ]; then
  git clone https://github.com/ggroup-ai-lab/gipformer.git
fi


In [ ]:
!pip install -q sentencepiece

import sentencepiece as spm
from pathlib import Path

ZIP_DIR = Path("/content/models/hynt_zipformer_30m")
bpe_model_path = ZIP_DIR / "bpe.model"
tokens_path = ZIP_DIR / "tokens.txt"

if bpe_model_path.exists():
    sp = spm.SentencePieceProcessor()
    sp.load(str(bpe_model_path))

    # Ghi file tokens.txt theo định dạng: <token> <id>
    with open(tokens_path, 'w', encoding='utf-8') as f:
        for i in range(sp.get_piece_size()):
            token = sp.id_to_piece(i)
            f.write(f"{token} {i}\n")

    print(f"✅ Đã tạo thành công file tokens.txt tại: {tokens_path}")
    print(f"Tổng số token: {sp.get_piece_size()}")

    # Kiểm tra nội dung vài dòng đầu
    !head -n 5 {tokens_path}
else:
    print("❌ LỖI: Không tìm thấy file bpe.model tại", bpe_model_path)

In [ ]:
# CELL 9
import os
import re
import json
import subprocess
import unicodedata
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import librosa

DENOISE_MANIFEST = Path("/content/drive/MyDrive/data_label_auto/manifests/denoised_segments.csv")
denoise_df = pd.read_csv(DENOISE_MANIFEST)

ZIP_DIR = Path("/content/models/hynt_zipformer_30m")
GIP_REPO = Path("/content/gipformer")

def normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def run_cmd(cmd):
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return p.returncode, p.stdout

def transcribe_zipformer_onnx(audio_path: str) -> str:
    """
    Dùng sherpa-onnx với model ONNX của hynt.
    """
    import sherpa_onnx

    recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
        encoder=str(ZIP_DIR / "encoder-epoch-20-avg-10.onnx"),
        decoder=str(ZIP_DIR / "decoder-epoch-20-avg-10.onnx"),
        joiner=str(ZIP_DIR / "joiner-epoch-20-avg-10.onnx"),
        tokens=str(ZIP_DIR / "tokens.txt"),
        num_threads=2,
        sample_rate=16000,
        feature_dim=80,
        decoding_method="greedy_search",
        provider="cpu",
    )

    stream = recognizer.create_stream()
    # Sử dụng librosa để load và tự động resample về 16000Hz
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    stream.accept_waveform(16000, audio)
    recognizer.decode_stream(stream)
    return stream.result.text.strip()

def transcribe_gipformer(audio_path: str) -> str:
    """
    Gọi infer_onnx.py của gipformer rồi parse stdout.
    """
    cmd = [
        "python",
        str(GIP_REPO / "infer_onnx.py"),
        "--audio",
        audio_path,
    ]
    code, out = run_cmd(cmd)
    if code != 0:
        raise RuntimeError(out)

    lines = [x.strip() for x in out.splitlines() if x.strip()]
    for ln in lines:
        if ln.startswith("Text:"):
            return ln.replace("Text:", "", 1).strip()

    return ""

# test 1 file
sample_audio = denoise_df.iloc[0]["denoised_path"]
print("sample:", sample_audio)
try:
    print("zipformer:", transcribe_zipformer_onnx(sample_audio))
except Exception as e:
    print("zipformer error:", e)

try:
    print("gipformer:", transcribe_gipformer(sample_audio))
except Exception as e:
    print("gipformer error:", e)


In [ ]:
# CELL 10
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import importlib
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import sherpa_onnx
import soundfile as sf
from tqdm.auto import tqdm


def ensure_onnxruntime_gpu(auto_fix: bool = True):
    """Load onnxruntime and try to ensure CUDA provider is available in Colab."""
    ort = None
    try:
        import onnxruntime as _ort
        ort = _ort
    except ModuleNotFoundError:
        if auto_fix:
            print("onnxruntime not found -> installing onnxruntime-gpu...")
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "-U", "onnxruntime-gpu"],
                check=False,
            )
            import onnxruntime as _ort
            ort = _ort
        else:
            raise

    providers = ort.get_available_providers()

    # If GPU exists but CUDA provider missing, try reinstalling gpu wheel once.
    if auto_fix and "CUDAExecutionProvider" not in providers and shutil.which("nvidia-smi"):
        print("CUDAExecutionProvider missing -> reinstalling onnxruntime-gpu...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "onnxruntime-gpu"],
            check=False,
        )
        importlib.invalidate_caches()
        if "onnxruntime" in sys.modules:
            del sys.modules["onnxruntime"]
        import onnxruntime as _ort
        ort = _ort
        providers = ort.get_available_providers()

    return ort, providers


def normalize_text(s: str) -> str:
    s = str(s or "").lower().strip()
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_audio_16k(audio_path: str, sample_rate: int) -> np.ndarray:
    wav, sr = sf.read(audio_path, dtype="float32")
    x = np.asarray(wav, dtype=np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if sr != sample_rate:
        x = librosa.resample(x, orig_sr=sr, target_sr=sample_rate).astype(np.float32)
    if len(x) == 0:
        return np.zeros(0, dtype=np.float32)
    return np.clip(x, -1.0, 1.0)


def get_gpu_name() -> str:
    if not shutil.which("nvidia-smi"):
        return ""
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        ).strip()
        return out.splitlines()[0].strip() if out else ""
    except Exception:
        return ""


def get_gpu_mem_gb() -> float:
    if not shutil.which("nvidia-smi"):
        return 0.0
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip()
        mb = float(out.splitlines()[0].strip()) if out else 0.0
        return mb / 1024.0
    except Exception:
        return 0.0


def init_zip_recognizer(provider_name: str, sample_rate: int, num_threads: int):
    zip_dir = Path("/content/models/hynt_zipformer_30m")
    return sherpa_onnx.OfflineRecognizer.from_transducer(
        encoder=str(zip_dir / "encoder-epoch-20-avg-10.onnx"),
        decoder=str(zip_dir / "decoder-epoch-20-avg-10.onnx"),
        joiner=str(zip_dir / "joiner-epoch-20-avg-10.onnx"),
        tokens=str(zip_dir / "tokens.txt"),
        num_threads=num_threads,
        sample_rate=sample_rate,
        feature_dim=80,
        decoding_method="greedy_search",
        provider=provider_name,
    )


def init_gip_recognizer(provider_name: str, num_threads: int):
    if "/content/gipformer" not in sys.path:
        sys.path.append("/content/gipformer")
    from infer_onnx import download_model, create_recognizer

    model_paths = download_model(quantize="fp32")
    try:
        rec = create_recognizer(model_paths, num_threads=num_threads, provider=provider_name)
        return rec, provider_name
    except TypeError:
        rec = create_recognizer(model_paths, num_threads=num_threads)
        return rec, "default"


def decode_texts_batch(recognizer, audios, sample_rate: int, batch_size: int):
    """Batch decode with auto fallback when batch is too large."""
    if recognizer is None:
        return [""] * len(audios), batch_size

    texts = [""] * len(audios)
    use_batch_api = hasattr(recognizer, "decode_streams")
    bs = max(1, int(batch_size))
    idx = 0

    while idx < len(audios):
        end = min(idx + bs, len(audios))
        part = audios[idx:end]

        streams = []
        for a in part:
            st = recognizer.create_stream()
            st.accept_waveform(sample_rate, a.astype(np.float32, copy=False))
            streams.append(st)

        try:
            if use_batch_api:
                recognizer.decode_streams(streams)
            else:
                for st in streams:
                    recognizer.decode_stream(st)

            for j, st in enumerate(streams):
                texts[idx + j] = str(getattr(st.result, "text", "") or "").strip()
            idx = end
        except Exception as e:
            msg = str(e).lower()
            if bs > 1 and any(k in msg for k in ["cuda", "memory", "cublas", "alloc", "out of"]):
                new_bs = max(1, bs // 2)
                print(f"[ASR] batch {bs} failed -> reduce to {new_bs}. err={e}")
                bs = new_bs
                continue

            if bs > 1:
                print(f"[ASR] batch {bs} failed (non-OOM) -> fallback bs=1. err={e}")
                bs = 1
                continue

            # bs == 1 and still fail: mark empty and continue
            print(f"[ASR] single decode failed at idx={idx}: {e}")
            texts[idx] = ""
            idx += 1

    return texts, bs


def append_rows_to_csv(rows, csv_path: Path):
    if not rows:
        return
    df = pd.DataFrame(rows)
    header = (not csv_path.exists()) or (csv_path.stat().st_size == 0)
    df.to_csv(csv_path, mode="a", header=header, index=False)


# =====================
# Paths
# =====================
BASE_DIR = Path("/content/drive/MyDrive/data_label_auto")
ACCEPT_DIR = BASE_DIR / "accepted"
REJECT_DIR = BASE_DIR / "rejected"
MANIFEST_DIR = BASE_DIR / "manifests"
DENOISE_MANIFEST = MANIFEST_DIR / "denoised_segments.csv"
ALL_RESULTS_CSV = MANIFEST_DIR / "all_results.csv"
ACCEPTED_CSV = MANIFEST_DIR / "accepted_manifest.csv"
ACCEPTED_JSONL = MANIFEST_DIR / "accepted_manifest.jsonl"

ACCEPT_DIR.mkdir(parents=True, exist_ok=True)
REJECT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Runtime config
# =====================
SAMPLE_RATE = 16000
SAVE_EVERY = 500
RESUME_FROM_ALL_RESULTS = True
FORCE_REPROCESS = False
MAX_ITEMS_THIS_RUN = None      # Example: 20000
MAX_RUN_MINUTES = None         # Example: 120
WRITE_ACCEPT_REJECT_COPY = False

# CPU/GPU tuning
ZIP_NUM_THREADS_CPU = 4
GIP_NUM_THREADS_CPU = 4
ZIP_NUM_THREADS_CUDA = 2
GIP_NUM_THREADS_CUDA = 2
ASR_BATCH_SIZE_CPU = 6
ASR_BATCH_SIZE_CUDA = 96       # T4 16GB start point, auto-reduce on OOM

# =====================
# Provider init
# =====================
ort, providers = ensure_onnxruntime_gpu(auto_fix=True)
use_cuda = "CUDAExecutionProvider" in providers
asr_provider = "cuda" if use_cuda else "cpu"
gpu_name = get_gpu_name()
gpu_mem_gb = get_gpu_mem_gb()

if use_cuda and gpu_mem_gb >= 15:
    ASR_BATCH_SIZE_CUDA = max(ASR_BATCH_SIZE_CUDA, 96)

effective_batch_size = ASR_BATCH_SIZE_CUDA if use_cuda else ASR_BATCH_SIZE_CPU
zip_threads = ZIP_NUM_THREADS_CUDA if use_cuda else ZIP_NUM_THREADS_CPU
gip_threads = GIP_NUM_THREADS_CUDA if use_cuda else GIP_NUM_THREADS_CPU

print("ONNX providers:", providers)
print("ASR provider:", asr_provider)
if gpu_name:
    print("GPU:", gpu_name, f"({gpu_mem_gb:.1f} GB)")
print("BATCH_SIZE:", effective_batch_size)
print("SAVE_EVERY:", SAVE_EVERY)
print("RESUME_FROM_ALL_RESULTS:", RESUME_FROM_ALL_RESULTS)
print("FORCE_REPROCESS:", FORCE_REPROCESS)
print("WRITE_ACCEPT_REJECT_COPY:", WRITE_ACCEPT_REJECT_COPY)

# Reuse global recognizers if already initialized
if "zip_recognizer" in globals() and zip_recognizer is not None:
    zip_asr = zip_recognizer
    print("Zipformer recognizer: reuse from globals")
else:
    print("Zipformer recognizer: init new")
    zip_asr = init_zip_recognizer(asr_provider, SAMPLE_RATE, zip_threads)

if "gip_recognizer" in globals() and gip_recognizer is not None:
    gip_asr = gip_recognizer
    print("Gipformer recognizer: reuse from globals")
else:
    print("Gipformer recognizer: init new")
    try:
        gip_asr, gip_provider_used = init_gip_recognizer(asr_provider, gip_threads)
        print("Gipformer provider used:", gip_provider_used)
    except Exception as e:
        gip_asr = None
        print(f"Failed to initialize Gipformer recognizer: {e}")

if not DENOISE_MANIFEST.exists():
    raise FileNotFoundError(f"Missing file: {DENOISE_MANIFEST}")

denoise_df = pd.read_csv(DENOISE_MANIFEST)
if denoise_df.empty:
    raise RuntimeError("denoised_segments.csv is empty")
if "denoised_path" not in denoise_df.columns:
    raise ValueError("denoised_segments.csv missing 'denoised_path' column")

for col in ["src", "duration_sec", "start_sec", "end_sec"]:
    if col not in denoise_df.columns:
        denoise_df[col] = np.nan

# Resume: skip already processed denoised_path
processed_set = set()
if RESUME_FROM_ALL_RESULTS and ALL_RESULTS_CSV.exists() and not FORCE_REPROCESS:
    try:
        done_df = pd.read_csv(ALL_RESULTS_CSV, usecols=["denoised_path"])
        processed_set = set(done_df["denoised_path"].astype(str).tolist())
    except Exception as e:
        print(f"Cannot load processed set from all_results.csv: {e}")

rows_all = denoise_df.to_dict("records")
if FORCE_REPROCESS:
    pending_rows = rows_all
else:
    pending_rows = [r for r in rows_all if str(r["denoised_path"]) not in processed_set]

if MAX_ITEMS_THIS_RUN is not None:
    pending_rows = pending_rows[: int(MAX_ITEMS_THIS_RUN)]

print("Total rows in denoise manifest:", len(rows_all))
print("Already processed (resume):", len(processed_set))
print("Pending this run:", len(pending_rows))

run_ok = 0
run_fail = 0
processed_now = 0
out_buffer = []
zip_bs = int(effective_batch_size)
gip_bs = int(effective_batch_size)
start_ts = time.time()
stop_early = False

pbar = tqdm(total=len(pending_rows), desc="ASR+Voting")
for start in range(0, len(pending_rows), max(1, int(effective_batch_size))):
    if MAX_RUN_MINUTES is not None:
        if (time.time() - start_ts) / 60.0 >= float(MAX_RUN_MINUTES):
            stop_early = True
            print(f"Reached MAX_RUN_MINUTES={MAX_RUN_MINUTES}. Stop safely for resume.")
            break

    chunk = pending_rows[start : start + max(1, int(effective_batch_size))]

    audios = []
    valid_idx = []
    for i, row in enumerate(chunk):
        wav_path = str(row["denoised_path"])
        try:
            aud = load_audio_16k(wav_path, SAMPLE_RATE)
            audios.append(aud)
            valid_idx.append(i)
        except Exception as e:
            row_local = dict(row)
            row_local.update(
                {
                    "zipformer_text": "",
                    "gipformer_text": "",
                    "zipformer_norm": "",
                    "gipformer_norm": "",
                    "decision_reason": f"audio_read_error:{e}",
                    "accepted": False,
                    "final_wav": "",
                    "final_text": "",
                }
            )
            out_buffer.append(row_local)
            run_fail += 1
            processed_now += 1
            pbar.update(1)

    if not valid_idx:
        if SAVE_EVERY > 0 and processed_now > 0 and processed_now % SAVE_EVERY == 0:
            append_rows_to_csv(out_buffer, ALL_RESULTS_CSV)
            out_buffer = []
            print(f"Checkpoint: processed_now={processed_now}")
        continue

    zip_texts, zip_bs = decode_texts_batch(zip_asr, audios, SAMPLE_RATE, zip_bs)
    gip_texts, gip_bs = decode_texts_batch(gip_asr, audios, SAMPLE_RATE, gip_bs) if gip_asr is not None else ([""] * len(audios), gip_bs)

    for local_i, row_i in enumerate(valid_idx):
        row = dict(chunk[row_i])
        wav_path = str(row["denoised_path"])

        zip_text = str(zip_texts[local_i] or "")
        gip_text = str(gip_texts[local_i] or "")
        zip_norm = normalize_text(zip_text)
        gip_norm = normalize_text(gip_text)

        accept = bool(zip_norm) and (zip_norm == gip_norm)
        if not zip_norm and not gip_norm:
            reason = "both_empty"
        elif not zip_norm:
            reason = "zip_empty"
        elif not gip_norm:
            reason = "gip_empty"
        elif accept:
            reason = "exact"
        else:
            reason = "mismatch"

        if WRITE_ACCEPT_REJECT_COPY:
            src_path = Path(wav_path)
            dst_path = (ACCEPT_DIR if accept else REJECT_DIR) / src_path.name
            if not dst_path.exists():
                shutil.copy2(src_path, dst_path)
            final_wav = str(dst_path)
        else:
            final_wav = wav_path if accept else ""

        row.update(
            {
                "zipformer_text": zip_text,
                "gipformer_text": gip_text,
                "zipformer_norm": zip_norm,
                "gipformer_norm": gip_norm,
                "decision_reason": reason,
                "accepted": bool(accept),
                "final_wav": final_wav,
                "final_text": zip_norm if accept else "",
            }
        )

        out_buffer.append(row)
        if accept:
            run_ok += 1
        else:
            run_fail += 1

        processed_now += 1
        pbar.update(1)

        if SAVE_EVERY > 0 and processed_now > 0 and processed_now % SAVE_EVERY == 0:
            append_rows_to_csv(out_buffer, ALL_RESULTS_CSV)
            out_buffer = []
            print(f"Checkpoint: processed_now={processed_now}, zip_bs={zip_bs}, gip_bs={gip_bs}")

pbar.close()

# Final flush
append_rows_to_csv(out_buffer, ALL_RESULTS_CSV)
out_buffer = []

# Rebuild accepted manifest from all_results (dedupe by denoised_path)
if ALL_RESULTS_CSV.exists() and ALL_RESULTS_CSV.stat().st_size > 0:
    res_df = pd.read_csv(ALL_RESULTS_CSV)
    if "denoised_path" in res_df.columns:
        res_df = res_df.drop_duplicates(subset=["denoised_path"], keep="last")
        res_df.to_csv(ALL_RESULTS_CSV, index=False)
else:
    res_df = pd.DataFrame()

if res_df.empty:
    accepted_df = pd.DataFrame(columns=["final_wav", "final_text", "duration_sec", "src", "start_sec", "end_sec"])
else:
    if "accepted" not in res_df.columns:
        raise ValueError("all_results missing 'accepted' column")

    accepted_df = res_df[res_df["accepted"] == True].copy()
    keep_cols = ["final_wav", "final_text", "duration_sec", "src", "start_sec", "end_sec"]
    for c in keep_cols:
        if c not in accepted_df.columns:
            accepted_df[c] = np.nan
    accepted_df = accepted_df[keep_cols].copy()

accepted_df.to_csv(ACCEPTED_CSV, index=False)
with open(ACCEPTED_JSONL, "w", encoding="utf-8") as f:
    for r in accepted_df.to_dict("records"):
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\nSaved:")
print("-", ALL_RESULTS_CSV)
print("-", ACCEPTED_CSV)
print("-", ACCEPTED_JSONL)
print("Current run accepted:", run_ok)
print("Current run rejected:", run_fail)
print("Total rows in all_results:", len(res_df))
print("Total accepted in all_results:", len(accepted_df))
print("Final batch sizes -> zip:", zip_bs, "gip:", gip_bs)
print("Status:", "stopped_early_for_resume" if stop_early else "completed")

if not res_df.empty and "decision_reason" in res_df.columns:
    print("\nReason counts:")
    print(res_df["decision_reason"].value_counts(dropna=False).head(20).to_string())

display(accepted_df.head(20))


In [ ]:
# CELL 11
import sys
from pathlib import Path

import onnxruntime as ort
import sherpa_onnx

providers = ort.get_available_providers()
use_cuda = "CUDAExecutionProvider" in providers
zip_provider = "cuda" if use_cuda else "cpu"

print("ONNX providers:", providers)
print("Zipformer provider:", zip_provider)

print("Initializing Zipformer...")
ZIP_DIR = Path("/content/models/hynt_zipformer_30m")
zip_recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=str(ZIP_DIR / "encoder-epoch-20-avg-10.onnx"),
    decoder=str(ZIP_DIR / "decoder-epoch-20-avg-10.onnx"),
    joiner=str(ZIP_DIR / "joiner-epoch-20-avg-10.onnx"),
    tokens=str(ZIP_DIR / "tokens.txt"),
    num_threads=2,
    sample_rate=16000,
    feature_dim=80,
    decoding_method="greedy_search",
    provider=zip_provider,
)

print("Initializing Gipformer...")
sys.path.append("/content/gipformer")
try:
    from infer_onnx import download_model, create_recognizer

    model_paths = download_model(quantize="fp32")
    # Gipformer helper may or may not expose provider arg. Try GPU first, then fallback.
    try:
        gip_recognizer = create_recognizer(model_paths, num_threads=2, provider=zip_provider)
        print(f"Gipformer provider: {zip_provider}")
    except TypeError:
        gip_recognizer = create_recognizer(model_paths, num_threads=2)
        print("Gipformer provider: default (helper has no provider arg)")
except Exception as e:
    gip_recognizer = None
    print(f"Failed to initialize Gipformer: {e}")

print("Recognizers are ready.")



In [ ]:
# CELL 12
import json
import os
import re
import difflib
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

# Word-aligned break-and-continue voting (no min-duration/min-word filter)
SAMPLE_RATE = 16000
TAIL_PAD_SEC = 0.0
MERGE_GAP_SEC = 0.0

# Debug one-file mode
DEBUG_TARGET_FILE = None  # None -> process all rejected
DEBUG_RUN_ONLY_FIRST_MATCH = False
DEBUG_VERBOSE = False
DEBUG_NO_WRITE = False  # write audio/manifest outputs


def normalize_word_text(s: str) -> str:
    s = str(s or "").lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def run_recognizer(recognizer, audio: np.ndarray, sample_rate: int = SAMPLE_RATE):
    stream = recognizer.create_stream()
    stream.accept_waveform(sample_rate, audio.astype(np.float32, copy=False))
    recognizer.decode_stream(stream)
    result = stream.result
    return {
        "text": str(getattr(result, "text", "") or "").strip(),
        "tokens": list(getattr(result, "tokens", []) or []),
        "timestamps": list(getattr(result, "timestamps", []) or []),
    }


def safe_time(val, default=0.0):
    try:
        t = float(val)
    except Exception:
        return float(default)
    if not np.isfinite(t):
        return float(default)
    return t


def extract_zip_words_with_time(zip_tokens, zip_timestamps):
    words = []
    n = min(len(zip_tokens), len(zip_timestamps))
    if n == 0:
        return words

    cleaned = []
    for i in range(n):
        tok = str(zip_tokens[i] or "").strip()
        if not tok:
            continue
        if tok.startswith("<") and tok.endswith(">"):
            continue
        cleaned.append((i, tok))

    if not cleaned:
        return words

    def add_word(raw_word, start_t, end_t):
        norm_word = normalize_word_text(raw_word)
        if not norm_word:
            return
        end_t = max(float(end_t), float(start_t) + 1e-3)
        words.append(
            {
                "raw": str(raw_word),
                "norm": norm_word,
                "start": float(start_t),
                "end": float(end_t),
            }
        )

    has_spm_boundary = any(tok.startswith("\u2581") for _, tok in cleaned)

    # Mode A: token stream has no sentencepiece boundary mark -> each token is a word.
    if not has_spm_boundary:
        for k, (i, tok) in enumerate(cleaned):
            t_start = safe_time(zip_timestamps[i], 0.0)
            if k + 1 < len(cleaned):
                next_i = cleaned[k + 1][0]
                t_end = safe_time(zip_timestamps[next_i], t_start + 1e-3)
            elif i + 1 < n:
                t_end = safe_time(zip_timestamps[i + 1], t_start + 0.05)
            else:
                t_end = t_start + 0.05
            add_word(tok, t_start, t_end)
        return words

    # Mode B: sentencepiece-like stream with ? boundary marker.
    cur_pieces = []
    cur_start = None
    cur_end = None

    def flush_word():
        nonlocal cur_pieces, cur_start, cur_end
        if not cur_pieces or cur_start is None or cur_end is None:
            cur_pieces = []
            cur_start = None
            cur_end = None
            return
        raw_word = "".join(cur_pieces).strip()
        add_word(raw_word, cur_start, cur_end)
        cur_pieces = []
        cur_start = None
        cur_end = None

    for i, tok in cleaned:
        starts_new_word = tok.startswith("\u2581")
        piece = tok.lstrip("\u2581")
        if not piece:
            continue

        t_start = safe_time(zip_timestamps[i], cur_end if cur_end is not None else 0.0)
        if i + 1 < n:
            t_end = safe_time(zip_timestamps[i + 1], t_start)
        else:
            t_end = t_start + 0.05

        if starts_new_word and cur_pieces:
            flush_word()

        if cur_start is None:
            cur_start = t_start
        cur_end = max(t_end, t_start + 1e-3)
        cur_pieces.append(piece)

    flush_word()

    # Safety fallback: if parsing collapsed too much, fallback to per-token words.
    if len(words) <= 1 and len(cleaned) >= 5:
        words = []
        for k, (i, tok) in enumerate(cleaned):
            t_start = safe_time(zip_timestamps[i], 0.0)
            if k + 1 < len(cleaned):
                next_i = cleaned[k + 1][0]
                t_end = safe_time(zip_timestamps[next_i], t_start + 1e-3)
            elif i + 1 < n:
                t_end = safe_time(zip_timestamps[i + 1], t_start + 0.05)
            else:
                t_end = t_start + 0.05
            add_word(tok.lstrip("\u2581"), t_start, t_end)

    return words

def find_consensus_runs_word_alignment(zip_tokens, zip_timestamps, gip_text, tail_pad_sec: float = TAIL_PAD_SEC):
    zip_words = extract_zip_words_with_time(zip_tokens, zip_timestamps)
    gip_words = [w for w in normalize_word_text(gip_text).split() if w]

    if not zip_words or not gip_words:
        return [], 0, 0, 0, [], zip_words, gip_words

    z_norm = [w["norm"] for w in zip_words]
    g_norm = gip_words

    sm = difflib.SequenceMatcher(a=z_norm, b=g_norm, autojunk=False)
    opcodes = sm.get_opcodes()

    runs = []
    events = []
    matched_words = 0
    mismatch_words = 0

    for tag, i1, i2, j1, j2 in opcodes:
        if tag == "equal":
            span = i2 - i1
            matched_words += span
            if span <= 0:
                continue

            start_time = float(zip_words[i1]["start"])
            end_time = float(zip_words[i2 - 1]["end"]) + tail_pad_sec
            end_time = max(end_time, start_time + 1e-3)

            text = " ".join(w["raw"] for w in zip_words[i1:i2]).strip()
            if text:
                runs.append(
                    {
                        "start_time": start_time,
                        "end_time": end_time,
                        "duration": round(end_time - start_time, 3),
                        "text": text,
                        "start_word_idx": int(i1),
                        "end_word_idx": int(i2 - 1),
                    }
                )

            events.append(
                {
                    "type": "equal",
                    "zip_word_span": [int(i1), int(i2)],
                    "gip_word_span": [int(j1), int(j2)],
                    "text": text,
                }
            )
        else:
            mismatch_words += max(i2 - i1, j2 - j1)
            events.append(
                {
                    "type": "break",
                    "tag": tag,
                    "zip_word_span": [int(i1), int(i2)],
                    "gip_word_span": [int(j1), int(j2)],
                    "zip_words": " ".join(z_norm[i1:i2]),
                    "gip_words": " ".join(g_norm[j1:j2]),
                }
            )

    aligned_words = matched_words + mismatch_words
    return runs, matched_words, mismatch_words, aligned_words, events, zip_words, gip_words


def merge_runs(audio: np.ndarray, sample_rate: int, runs, gap_sec: float = MERGE_GAP_SEC):
    packed = []
    for run in runs:
        s = max(0, int(run["start_time"] * sample_rate))
        e = min(len(audio), int(run["end_time"] * sample_rate))
        if e <= s:
            continue
        clip = audio[s:e]
        if clip.size:
            packed.append((clip.astype(np.float32, copy=False), run["text"], run))

    if not packed:
        return None

    gap = np.zeros(int(gap_sec * sample_rate), dtype=np.float32) if gap_sec > 0 else None
    merged_parts = []
    merged_text_parts = []
    used_runs = []

    for i, (clip, text, run) in enumerate(packed):
        merged_parts.append(clip)
        merged_text_parts.append(text)
        used_runs.append(run)
        if gap is not None and i < len(packed) - 1:
            merged_parts.append(gap)

    merged_audio = np.concatenate(merged_parts)
    merged_text = " ".join(t for t in merged_text_parts if str(t).strip()).strip()
    return merged_audio, merged_text, used_runs


if (
    "zip_recognizer" not in globals()
    or zip_recognizer is None
    or "gip_recognizer" not in globals()
    or gip_recognizer is None
):
    raise RuntimeError("Run recognizer initialization cell before this voting cell.")

ACCEPT_DIR = Path("/content/drive/MyDrive/data_label_auto/accepted")
MANIFEST_DIR = Path("/content/drive/MyDrive/data_label_auto/manifests")
results_csv = MANIFEST_DIR / "all_results.csv"

ACCEPT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

df_all = pd.read_csv(results_csv)
acc_mask = df_all.get("accepted", False)
if not isinstance(acc_mask, pd.Series):
    acc_mask = pd.Series([bool(acc_mask)] * len(df_all), index=df_all.index)
acc_mask = acc_mask.astype(str).str.lower().isin(["true", "1", "yes"])
rejected_df = df_all[~acc_mask].copy()

process_df = rejected_df.copy()
if DEBUG_TARGET_FILE:
    process_df = process_df[
        process_df["denoised_path"].astype(str).str.contains(DEBUG_TARGET_FILE, regex=False, na=False)
    ].copy()
    if DEBUG_RUN_ONLY_FIRST_MATCH and len(process_df) > 1:
        process_df = process_df.head(1)

all_salvaged_rows = []
all_run_rows = []
file_debug_rows = []

print("")
print(f"Start word-aligned break-and-continue on {len(process_df)} selected file(s)...")
if DEBUG_TARGET_FILE:
    print("DEBUG_TARGET_FILE:", DEBUG_TARGET_FILE)

for _, row in tqdm(process_df.iterrows(), total=len(process_df)):
    wav_path = row.get("denoised_path", "")
    if not wav_path or not os.path.exists(wav_path):
        continue

    try:
        audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
        if audio.size == 0:
            continue

        zip_res = run_recognizer(zip_recognizer, audio)
        gip_res = run_recognizer(gip_recognizer, audio)

        runs, matched_words, mismatch_words, aligned_words, events, zip_words, gip_words = find_consensus_runs_word_alignment(
            zip_res["tokens"],
            zip_res["timestamps"],
            gip_res["text"],
            tail_pad_sec=TAIL_PAD_SEC,
        )

        consensus_text = " ".join(r["text"] for r in runs if r.get("text"))

        file_debug_rows.append(
            {
                "wav": wav_path,
                "zip_words": len(zip_words),
                "gip_words": len(gip_words),
                "aligned_words": aligned_words,
                "matched_words": matched_words,
                "mismatch_words": mismatch_words,
                "num_runs": len(runs),
                "consensus_text": consensus_text,
                "events_json": json.dumps(events, ensure_ascii=False),
            }
        )

        if DEBUG_VERBOSE:
            print("=" * 120)
            print("FILE:", wav_path)
            print(f"aligned_words={aligned_words} matched_words={matched_words} mismatch_words={mismatch_words} runs={len(runs)}")
            print("ZIP_TEXT_FULL:")
            print(zip_res.get("text", ""))
            print("GIP_TEXT_FULL:")
            print(gip_res.get("text", ""))
            print("CONSENSUS_TEXT:")
            print(consensus_text)
            if runs:
                print("RUNS:")
                for ridx, run in enumerate(runs):
                    print(
                        f"  - run_{ridx}: word_idx[{run['start_word_idx']}:{run['end_word_idx']}] "
                        f"time[{run['start_time']:.2f},{run['end_time']:.2f}] "
                        f"dur={run['duration']:.2f}s text={run['text']}"
                    )
            else:
                print("RUNS: <none>")

        if not runs:
            continue

        merged = merge_runs(audio, SAMPLE_RATE, runs, gap_sec=MERGE_GAP_SEC)
        if not merged:
            continue

        merged_audio, merged_text, used_runs = merged
        merged_duration = len(merged_audio) / SAMPLE_RATE
        if not merged_text or merged_duration <= 0:
            continue

        out_name = f"salvaged_vote_{Path(wav_path).name}"
        out_path = ACCEPT_DIR / out_name

        if not DEBUG_NO_WRITE:
            sf.write(str(out_path), merged_audio, SAMPLE_RATE)

        new_row = row.copy()
        new_row["final_wav"] = str(out_path)
        new_row["final_text"] = merged_text
        new_row["duration_sec"] = round(merged_duration, 3)
        new_row["accepted"] = True
        new_row["is_salvaged_vote"] = True
        new_row["num_merged_runs"] = len(used_runs)
        new_row["matched_words"] = matched_words
        new_row["mismatch_words"] = mismatch_words
        new_row["merged_runs"] = json.dumps(used_runs, ensure_ascii=False)
        all_salvaged_rows.append(new_row)

        for run_idx, run in enumerate(used_runs):
            all_run_rows.append(
                {
                    "src_wav": wav_path,
                    "out_wav": str(out_path),
                    "run_idx": run_idx,
                    "start_word_idx": run["start_word_idx"],
                    "end_word_idx": run["end_word_idx"],
                    "start_time": run["start_time"],
                    "end_time": run["end_time"],
                    "duration": run["duration"],
                    "text": run["text"],
                }
            )

    except Exception as e:
        print(f"Error processing {Path(wav_path).name}: {e}")

if DEBUG_TARGET_FILE:
    out_csv = MANIFEST_DIR / "salvaged_vote_manifest_debug_onefile.csv"
    run_csv = MANIFEST_DIR / "salvaged_vote_runs_manifest_debug_onefile.csv"
    debug_csv = MANIFEST_DIR / "salvaged_vote_debug_onefile.csv"
else:
    out_csv = MANIFEST_DIR / "salvaged_vote_manifest.csv"
    run_csv = MANIFEST_DIR / "salvaged_vote_runs_manifest.csv"
    debug_csv = MANIFEST_DIR / "salvaged_vote_debug.csv"

if file_debug_rows:
    pd.DataFrame(file_debug_rows).to_csv(debug_csv, index=False)
    print(f"Saved debug: {debug_csv}")

if all_salvaged_rows:
    salvaged_df = pd.DataFrame(all_salvaged_rows)

    if not DEBUG_NO_WRITE:
        salvaged_df.to_csv(out_csv, index=False)
        if all_run_rows:
            pd.DataFrame(all_run_rows).to_csv(run_csv, index=False)
            print(f"Saved run details: {run_csv}")

    print("")
    print(f"Salvaged {len(salvaged_df)} files with word-aligned break-and-continue voting.")
    if DEBUG_NO_WRITE:
        print("DEBUG_NO_WRITE=True -> skip writing audio/manifest")
    else:
        print(f"Saved manifest: {out_csv}")
    display(salvaged_df[["final_wav", "final_text", "duration_sec", "num_merged_runs", "matched_words", "mismatch_words"]].head(10))
else:
    print("")
    print("No file produced merged output in current debug run.")



In [ ]:
# CELL 13
import pandas as pd
import soundfile as sf
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/data_label_auto")
MANIFEST_DIR = BASE_DIR / "manifests"
RAW_DIR = BASE_DIR / "raw"

all_path = MANIFEST_DIR / "all_results.csv"
accepted_pre_path = MANIFEST_DIR / "accepted_manifest.csv"  # pre-salvage output from CELL 10
accepted_combined_path = MANIFEST_DIR / "accepted_manifest_combined.csv"
salvaged_path = MANIFEST_DIR / "salvaged_vote_manifest.csv"
vad_path = MANIFEST_DIR / "vad_segments.csv"
denoise_path = MANIFEST_DIR / "denoised_segments.csv"

if not all_path.exists():
    raise FileNotFoundError(f"Missing required file: {all_path}. Run accept/reject cell first.")

all_df = pd.read_csv(all_path)
keep_cols = ["final_wav", "final_text", "duration_sec", "src", "start_sec", "end_sec"]

# Rebuild pre-salvage accepted directly from all_results to avoid stale/overwritten manifest.
if "accepted" not in all_df.columns:
    raise ValueError("all_results.csv missing 'accepted' column")

accepted_pre_df = all_df[all_df["accepted"] == True].copy()
for c in keep_cols:
    if c not in accepted_pre_df.columns:
        accepted_pre_df[c] = None
accepted_pre_df = accepted_pre_df[keep_cols].copy()
accepted_pre_df.to_csv(accepted_pre_path, index=False)

sal_df = pd.read_csv(salvaged_path) if salvaged_path.exists() else pd.DataFrame()
for c in keep_cols:
    if c not in sal_df.columns:
        sal_df[c] = None
sal_df = sal_df[keep_cols].copy() if not sal_df.empty else pd.DataFrame(columns=keep_cols)

# Combine pre-accepted + salvaged, then deduplicate by output wav path.
combined_acc = pd.concat([accepted_pre_df, sal_df], ignore_index=True)
if not combined_acc.empty and "final_wav" in combined_acc.columns:
    combined_acc = combined_acc.drop_duplicates(subset=["final_wav"], keep="last")
combined_acc.to_csv(accepted_combined_path, index=False)


def safe_sum_hours(df, col="duration_sec"):
    if df is None or df.empty or col not in df.columns:
        return 0.0
    return float(pd.to_numeric(df[col], errors="coerce").fillna(0).sum() / 3600.0)

# Count raw hours without double-counting: prefer *_16k.wav; fallback to *.wav.
raw_files = sorted(RAW_DIR.glob("*_16k.wav"))
if not raw_files:
    raw_files = sorted(RAW_DIR.glob("*.wav"))

raw_hours = 0.0
for wav in raw_files:
    try:
        raw_hours += float(sf.info(str(wav)).duration) / 3600.0
    except Exception:
        pass

vad_df = pd.read_csv(vad_path) if vad_path.exists() else pd.DataFrame()
den_df = pd.read_csv(denoise_path) if denoise_path.exists() else pd.DataFrame()

print("=== Pipeline Hours Breakdown ===")
print("Raw input hours        :", round(raw_hours, 3))
print("VAD segment hours      :", round(safe_sum_hours(vad_df), 3))
print("Denoised segment hours :", round(safe_sum_hours(den_df), 3))
print("Accepted (pre-salvage) :", round(safe_sum_hours(accepted_pre_df), 3))
print("Salvaged voting hours  :", round(safe_sum_hours(sal_df), 3))
print("Accepted (combined)    :", round(safe_sum_hours(combined_acc), 3))

print("")
print("=== Acceptance Stats ===")
print("Total segments :", len(all_df))
print("Accepted       :", len(combined_acc))
print("Accept rate    :", round(100 * len(combined_acc) / max(1, len(all_df)), 2), "%")

print("")
print("Saved manifests:")
print("-", accepted_pre_path)
print("-", accepted_combined_path)

if len(combined_acc) and "duration_sec" in combined_acc.columns:
    print("Total accepted hours:", round(combined_acc["duration_sec"].sum() / 3600, 3))
    display(combined_acc.sample(min(10, len(combined_acc))))



In [ ]:
# CELL 14
import os
import difflib
from pathlib import Path

import librosa
import numpy as np
import pandas as pd

# Deep debug config
MANIFEST_DIR = Path("/content/drive/MyDrive/data_label_auto/manifests")
ALL_PATH = MANIFEST_DIR / "all_results.csv"
DEBUG_DIR = MANIFEST_DIR / "reject_debug_logs"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

DEBUG_TARGET_FILE = "a1us4y0oc10_16k__seg_00503_7597.91_7617.91.wav"
DEBUG_TOP_N = 5                # used only when DEBUG_TARGET_FILE is None
DEBUG_REASON = "mismatch"      # set None to include all reasons
SORT_BY_DURATION_DESC = True
RERUN_WITH_CURRENT_ALGO = True  # needs CELL 11 + CELL 12 already run

pd.set_option("display.max_colwidth", None)

if not ALL_PATH.exists():
    raise FileNotFoundError(f"Missing file: {ALL_PATH}")

df = pd.read_csv(ALL_PATH)
for c in [
    "accepted", "decision_reason", "duration_sec", "denoised_path",
    "zipformer_text", "gipformer_text", "zipformer_norm", "gipformer_norm",
]:
    if c not in df.columns:
        df[c] = ""

df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors="coerce").fillna(0.0)
accepted_mask = df["accepted"].astype(str).str.lower().isin(["true", "1", "yes"])
rej = df[~accepted_mask].copy()

if DEBUG_TARGET_FILE:
    rej = rej[
        rej["denoised_path"].astype(str).str.contains(DEBUG_TARGET_FILE, regex=False, na=False)
    ].copy()

if DEBUG_REASON is not None:
    rej = rej[rej["decision_reason"] == DEBUG_REASON].copy()

if DEBUG_TARGET_FILE:
    rej = rej.head(1)
else:
    rej = rej.sort_values("duration_sec", ascending=not SORT_BY_DURATION_DESC).head(DEBUG_TOP_N)

print("=== Deep Reject Debug ===")
print("Source file:", ALL_PATH)
print("DEBUG_TARGET_FILE:", DEBUG_TARGET_FILE)
print("Num rejected selected:", len(rej))
print("Reason filter:", DEBUG_REASON)
print("Debug logs dir:", DEBUG_DIR)


def normalize_text(s: str) -> str:
    s = str(s or "").lower().strip()
    s = " ".join(s.split())
    return s


def split_words(s: str):
    return [w for w in str(s or "").split() if w]


def diff_ops(a_words, b_words):
    sm = difflib.SequenceMatcher(a=a_words, b=b_words, autojunk=False)
    ops = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        left = " ".join(a_words[i1:i2]) if i2 > i1 else "<empty>"
        right = " ".join(b_words[j1:j2]) if j2 > j1 else "<empty>"
        ops.append(f"{tag}: zip[{i1}:{i2}]={left} | gip[{j1}:{j2}]={right}")
    return sm.ratio(), ops


def build_case_report(row, case_idx, total_cases):
    wav_path = str(row.get("denoised_path", ""))
    wav_name = os.path.basename(wav_path)
    duration = float(row.get("duration_sec", 0.0))
    reason = str(row.get("decision_reason", ""))

    zip_raw = str(row.get("zipformer_text", "") or "")
    gip_raw = str(row.get("gipformer_text", "") or "")

    zip_norm = str(row.get("zipformer_norm", "") or "")
    gip_norm = str(row.get("gipformer_norm", "") or "")
    if not zip_norm:
        zip_norm = normalize_text(zip_raw)
    if not gip_norm:
        gip_norm = normalize_text(gip_raw)

    z_words = split_words(zip_norm)
    g_words = split_words(gip_norm)
    ratio, ops = diff_ops(z_words, g_words)

    lines = []
    lines.append("=" * 120)
    lines.append(f"CASE {case_idx}/{total_cases}: {wav_name}")
    lines.append(f"reason={reason} duration_sec={duration:.2f}")
    lines.append("")

    lines.append("[STEP 1] FULL RAW TEXT FROM CSV (NO TRUNCATION)")
    lines.append("ZIP_RAW:")
    lines.append(zip_raw)
    lines.append("GIP_RAW:")
    lines.append(gip_raw)
    lines.append("")

    lines.append("[STEP 2] FULL NORMALIZED TEXT FROM CSV")
    lines.append(f"zip_norm == gip_norm: {zip_norm == gip_norm}")
    lines.append("ZIP_NORM:")
    lines.append(zip_norm)
    lines.append("GIP_NORM:")
    lines.append(gip_norm)
    lines.append("")

    lines.append("[STEP 3] WORD ALIGNMENT DIFF (FULL OPS)")
    lines.append(f"zip_words/gip_words: {len(z_words)}/{len(g_words)}")
    lines.append(f"token_ratio: {ratio:.6f}")
    if ops:
        lines.append("diff_ops:")
        for op in ops:
            lines.append(f"- {op}")
    else:
        lines.append("diff_ops: <none>")

    if RERUN_WITH_CURRENT_ALGO:
        lines.append("")
        lines.append("[STEP 4] RERUN CURRENT VOTING ALGO (CELL 12) WITH TIMESTAMPS")

        required = [
            "zip_recognizer", "gip_recognizer", "run_recognizer",
            "find_consensus_runs_word_alignment", "merge_runs", "SAMPLE_RATE", "MERGE_GAP_SEC",
        ]
        missing = [x for x in required if x not in globals()]
        if missing:
            lines.append(f"SKIP rerun: missing globals {missing}")
            lines.append("Run CELL 11 and CELL 12 before this debug cell.")
        elif not wav_path or not os.path.exists(wav_path):
            lines.append(f"SKIP rerun: missing audio file {wav_path}")
        else:
            try:
                audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
                z = run_recognizer(zip_recognizer, audio)
                g = run_recognizer(gip_recognizer, audio)

                lines.append("ZIP_RERUN_TEXT_FULL:")
                lines.append(str(z.get("text", "") or ""))
                lines.append("GIP_RERUN_TEXT_FULL:")
                lines.append(str(g.get("text", "") or ""))

                runs, matched_words, mismatch_words, aligned_words, events, zip_words, gip_words = find_consensus_runs_word_alignment(
                    z.get("tokens", []),
                    z.get("timestamps", []),
                    g.get("text", ""),
                    tail_pad_sec=TAIL_PAD_SEC if "TAIL_PAD_SEC" in globals() else 0.0,
                )

                lines.append(f"aligned_words={aligned_words} matched_words={matched_words} mismatch_words={mismatch_words}")
                lines.append(f"num_runs={len(runs)}")

                if runs:
                    lines.append("runs_detail:")
                    for ridx, run in enumerate(runs):
                        lines.append(
                            f"- run_{ridx}: [{run['start_time']:.2f}, {run['end_time']:.2f}] dur={run['duration']:.2f}s text={run['text']}"
                        )
                else:
                    lines.append("runs_detail: <none>")

                merged = merge_runs(audio, SAMPLE_RATE, runs, gap_sec=MERGE_GAP_SEC)
                if merged is None:
                    lines.append("merge_runs: None")
                else:
                    merged_audio, merged_text, used_runs = merged
                    merged_duration = len(merged_audio) / SAMPLE_RATE
                    lines.append(f"merged_duration_sec={merged_duration:.3f}")
                    lines.append(f"used_runs={len(used_runs)}")
                    lines.append("MERGED_TEXT_FULL:")
                    lines.append(str(merged_text or ""))
            except Exception as e:
                lines.append(f"RERUN ERROR: {e}")

    return lines


for idx, (_, row) in enumerate(rej.iterrows(), start=1):
    case_lines = build_case_report(row, idx, len(rej))
    report = "\n".join(case_lines)
    print(report)
    print("\n")

    stem = Path(str(row.get("denoised_path", f"case_{idx}"))).stem
    if DEBUG_TARGET_FILE:
        out_txt = DEBUG_DIR / f"debug_onefile_{stem}.txt"
    else:
        out_txt = DEBUG_DIR / f"debug_{idx:03d}_{stem}.txt"
    out_txt.write_text(report, encoding="utf-8")
    print("Saved:", out_txt)

print("\nDone deep debug. Open .txt files in:", DEBUG_DIR)


